# Week 3: Modular Thinking — Divide & Conquer — PHASE 2: The Decomposition Recipe

*Core Mastery: "I can decompose a problem into small, independent, testable functions"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Explain the **Single Responsibility Principle** for functions
2. Apply the **Decomposition Recipe**: Read → Decompose → Name → Contract → Draw → Code
3. Define **input/output contracts** for each function before coding
4. Draw a **dependency graph** showing which function calls which
5. Organize notebook cells in the **CONFIG → helpers → pipeline → output** pattern
6. Identify and **eliminate global variables** from a script
7. Use the **`main()` entry point** pattern for clean notebook execution
8. Practice **bottom-up implementation**: code leaf functions first, then compose
9. Refactor a monolithic 50-line script into modular functions
10. Test each module independently with assertions

## 🎯 Core Mastery Connection

In Week 1 you learned to refactor code for clarity. In Week 2 you learned to write professional functions with docstrings and defaults. This week we combine those skills with a *design methodology*: the **Decomposition Recipe**. Instead of writing code top-down in one big block, you will learn to break every problem into small pieces, define each piece’s contract, and assemble them bottom-up. This is how professional engineers build reliable software.

---
## Part 1: Why Decompose? The Single Responsibility Principle

The **Single Responsibility Principle (SRP)** states:

> Every function should do **one thing** and do it well.

### Signs a function does too much:
- It is longer than 10-15 lines
- It has multiple "sections" separated by blank lines or comments
- It is hard to name ("process_and_format_and_save_data")
- You cannot describe it in one sentence without using "and"

### Benefits of small functions:
| Benefit | Explanation |
|---------|-------------|
| **Testable** | Easy to write assertions for one responsibility |
| **Reusable** | A function that does one thing can be used anywhere |
| **Readable** | The function name describes its purpose |
| **Debuggable** | When something breaks, you know exactly where to look |

In [ ]:
# ❌ One function doing everything — Ahmet's weather report
def weather_report(temps, city):
    # validate
    clean = []
    for t in temps:
        if t is not None and -50 < t < 60:
            clean.append(t)
    # compute stats
    avg = sum(clean) / len(clean)
    hi = max(clean)
    lo = min(clean)
    # classify
    if avg > 30:
        status = "SICAK"
    elif avg > 15:
        status = "ILIK"
    else:
        status = "SOGUK"
    # format output
    print(f"{'='*40}")
    print(f"  Hava Durumu: {city}")
    print(f"{'='*40}")
    print(f"  Ortalama : {avg:.1f}°C")
    print(f"  En Yüksek: {hi:.1f}°C")
    print(f"  En Düşük : {lo:.1f}°C")
    print(f"  Durum    : {status}")
    print(f"{'='*40}")

weather_report([22, None, 28, 35, 31, -999, 27, 30], "Istanbul")

**Figure 1.1** — A monolithic function: validate + compute + classify + format.

In [ ]:
# ✅ Decomposed into single-responsibility functions
def clean_temperatures(raw_temps, low=-50, high=60):
    """Remove None and out-of-range values."""
    return [t for t in raw_temps if t is not None and low < t < high]

def compute_temp_stats(temps):
    """Return dict with mean, high, low."""
    return {
        "mean": sum(temps) / len(temps),
        "high": max(temps),
        "low": min(temps),
    }

def classify_temperature(avg_temp):
    """Classify average temperature in Turkish."""
    if avg_temp > 30: return "SICAK"
    if avg_temp > 15: return "ILIK"
    return "SOGUK"

def format_weather_report(city, stats, status):
    """Return a formatted weather report string."""
    lines = [
        "=" * 40,
        f"  Hava Durumu: {city}",
        "=" * 40,
        f"  Ortalama : {stats['mean']:.1f}°C",
        f"  En Yüksek: {stats['high']:.1f}°C",
        f"  En Düşük : {stats['low']:.1f}°C",
        f"  Durum    : {status}",
        "=" * 40,
    ]
    return "\n".join(lines)

# Pipeline
raw = [22, None, 28, 35, 31, -999, 27, 30]
cleaned = clean_temperatures(raw)
stats = compute_temp_stats(cleaned)
status = classify_temperature(stats["mean"])
report = format_weather_report("Istanbul", stats, status)
print(report)

**Figure 1.2** — Four small functions, each with a single responsibility, composed in a pipeline.

---
## Part 2: The Decomposition Recipe

When faced with a complex task, follow these six steps:

| Step | Action | Output |
|------|--------|--------|
| 1. **Read** | Understand the full problem | Problem statement in your own words |
| 2. **Decompose** | Break into sub-tasks | List of 4-8 sub-tasks |
| 3. **Name** | Give each sub-task a function name | `clean_data`, `compute_stats`, etc. |
| 4. **Contract** | Define inputs/outputs for each | `(list[float]) -> dict` |
| 5. **Draw** | Sketch the dependency graph | Which function calls which |
| 6. **Code** | Implement bottom-up | Start with leaf functions |

### Example: Bridge Load Analysis

**Problem:** Given a list of vehicle weights (some invalid), determine if the bridge is overloaded, and generate a report.

In [ ]:
# Step 1: READ — understand the problem
# "Given vehicle weights (may contain invalid values),
#  calculate total load, compare to bridge capacity,
#  and produce a status report."

# Step 2: DECOMPOSE — list sub-tasks
# 1. Validate and clean the weight data
# 2. Calculate total load
# 3. Compare load to bridge capacity
# 4. Generate a formatted report

# Step 3: NAME — function names
# clean_weights, total_load, check_capacity, bridge_report

# Step 4: CONTRACT — inputs and outputs
# clean_weights(raw: list) -> list[float]
# total_load(weights: list[float]) -> float
# check_capacity(load: float, capacity: float) -> str
# bridge_report(weights, load, status, capacity) -> str

print("Decomposition plan complete — ready to code!")

**Figure 2.1** — The first four steps of the Decomposition Recipe (planning phase).

In [ ]:
# Step 5: DRAW — dependency graph (as text)
print("""
Dependency Graph:
=================

  clean_weights()
        |
        v
   total_load()
        |
        v
  check_capacity()
        |
        v
  bridge_report()   <-- uses outputs from all above
""")

**Figure 2.2** — A text-based dependency graph showing the function pipeline.

In [ ]:
# Step 6: CODE — implement bottom-up (leaf functions first)

# CONFIG
BRIDGE_CAPACITY_KG = 50_000
MIN_VEHICLE_WEIGHT = 100     # kg — below this is probably an error
MAX_VEHICLE_WEIGHT = 40_000  # kg — above this is probably an error

def clean_weights(raw_weights):
    """Remove invalid weight values.

    Args:
        raw_weights: List of raw weight values (may contain None, negatives, outliers).

    Returns:
        List of valid weights within [MIN_VEHICLE_WEIGHT, MAX_VEHICLE_WEIGHT].
    """
    return [w for w in raw_weights
            if w is not None and MIN_VEHICLE_WEIGHT <= w <= MAX_VEHICLE_WEIGHT]

def total_load(weights):
    """Calculate the total weight load.

    Args:
        weights: List of valid weights in kg.

    Returns:
        Sum of all weights in kg.
    """
    return sum(weights)

def check_capacity(load, capacity=BRIDGE_CAPACITY_KG):
    """Compare load to bridge capacity.

    Args:
        load: Current total load in kg.
        capacity: Bridge weight limit in kg.

    Returns:
        Status string: 'SAFE', 'WARNING', or 'OVERLOADED'.
    """
    ratio = load / capacity
    if ratio > 1.0:
        return "OVERLOADED"
    elif ratio > 0.8:
        return "WARNING"
    return "SAFE"

def bridge_report(vehicles, load, status, capacity=BRIDGE_CAPACITY_KG):
    """Generate a formatted bridge status report.

    Args:
        vehicles: List of valid vehicle weights.
        load: Total load in kg.
        status: Status string from check_capacity.
        capacity: Bridge capacity in kg.

    Returns:
        Multi-line formatted report string.
    """
    pct = load / capacity * 100
    lines = [
        "=" * 45,
        "  BRIDGE LOAD REPORT",
        "=" * 45,
        f"  Vehicles  : {len(vehicles)}",
        f"  Total Load: {load:,.0f} kg",
        f"  Capacity  : {capacity:,.0f} kg",
        f"  Usage     : {pct:.1f}%",
        f"  Status    : {status}",
        "=" * 45,
    ]
    return "\n".join(lines)

# Test the pipeline
raw = [3500, None, 8200, -50, 12000, 15000, 7500, 4200, 99999]
valid = clean_weights(raw)
load = total_load(valid)
status = check_capacity(load)
print(bridge_report(valid, load, status))

**Figure 2.3** — Full implementation following the Decomposition Recipe, bottom-up.

---
## Part 3: Input/Output Contracts

An **I/O contract** defines:
- What a function **expects** (types, ranges, valid values)
- What it **returns** (type, meaning)
- What can go **wrong** (error conditions)

Write the contract *before* the implementation. Put it in the docstring.

| Function | Input Contract | Output Contract |
|----------|---------------|-----------------|
| `clean_weights(raw)` | `list` of any values | `list[float]` of valid weights |
| `total_load(weights)` | `list[float]`, non-empty | `float` ≥ 0 |
| `check_capacity(load, cap)` | two positive `float`s | `str` in {"SAFE", "WARNING", "OVERLOADED"} |

In [ ]:
# Enforcing contracts with assertions and checks
def safe_mean(values):
    """Calculate the mean of a non-empty list of numbers.

    Args:
        values: Non-empty list of numeric values.

    Returns:
        Arithmetic mean as float.

    Raises:
        ValueError: If values is empty.
        TypeError: If values contains non-numeric types.
    """
    if not values:
        raise ValueError("Cannot compute mean of empty list")
    if not all(isinstance(v, (int, float)) for v in values):
        raise TypeError("All values must be numeric")
    return sum(values) / len(values)

# Contract satisfied
print(safe_mean([10, 20, 30]))  # 20.0

# Contract violated
try:
    safe_mean([])
except ValueError as e:
    print(f"Caught: {e}")

try:
    safe_mean([10, "oops", 30])
except TypeError as e:
    print(f"Caught: {e}")

**Figure 3.1** — A function that explicitly checks its input contract.

In [ ]:
# Contract documentation with type hints (preview — we'll go deeper later)
def voltage_divider(vin: float, r1: float, r2: float) -> float:
    """Calculate output voltage of a voltage divider.

    Contract:
        - vin: any float (can be negative for AC)
        - r1, r2: positive floats (ohms)
        - returns: float in range [0, vin] (for positive vin)

    Args:
        vin: Input voltage in volts.
        r1: Top resistor in ohms (must be > 0).
        r2: Bottom resistor in ohms (must be > 0).

    Returns:
        Output voltage in volts.
    """
    assert r1 > 0 and r2 > 0, "Resistances must be positive"
    return vin * r2 / (r1 + r2)

print(f"Vout = {voltage_divider(12.0, 10000, 4700):.2f} V")

**Figure 3.2** — Type hints and assertions document the contract at a glance.

In [ ]:
# Designing contracts before coding — Zeynep's sensor pipeline
# Step: write empty functions with contracts (stubs)

def validate_sensor_reading(raw_value, sensor_range=(0, 100)):
    """Validate a single sensor reading.

    Args:
        raw_value: The raw reading (may be None or out of range).
        sensor_range: Tuple of (min, max) valid range.

    Returns:
        The reading if valid, or None if invalid.
    """
    pass  # TODO: implement

def aggregate_readings(valid_readings):
    """Compute summary statistics from valid readings.

    Args:
        valid_readings: Non-empty list of float values.

    Returns:
        Dict with keys 'mean', 'min', 'max', 'count'.
    """
    pass  # TODO: implement

def generate_alert(stats, thresholds):
    """Check statistics against thresholds and return alerts.

    Args:
        stats: Dict from aggregate_readings.
        thresholds: Dict with 'high' and 'low' keys.

    Returns:
        List of alert strings (empty if all OK).
    """
    pass  # TODO: implement

print("Contracts defined — implementation comes next!")

**Figure 3.3** — Stub functions with contracts defined before any implementation.

---
## Part 4: Dependency Graphs — Which Function Calls Which

A **dependency graph** shows the call relationships between functions.

### Rules for good dependency graphs:
- **No cycles** — function A should not call B which calls A
- **Leaf functions** have no dependencies (pure computation)
- **Pipeline functions** call other functions in sequence
- **The `main()` function** is the root node, calling the pipeline

### Reading the graph:
- Arrow from A → B means "A calls B"
- Leaf nodes (no outgoing arrows) are coded **first**
- Root nodes (no incoming arrows) are coded **last**

In [ ]:
# Visualizing a dependency graph as text
def print_dependency_graph():
    """Display the weather station dependency graph."""
    graph = """
    ┌─────────────────────────┐
    │        main()           │
    └────────┬────────────────┘
             │
    ┌────────v────────────────┐
    │   run_weather_pipeline()│
    └──┬──────┬──────┬────────┘
       │      │      │
       v      v      v
  ┌────────┐ ┌─────────┐ ┌──────────────┐
  │clean() │ │compute()│ │format_report()│
  └────────┘ └──┬──────┘ └──────────────┘
                │
                v
           ┌──────────┐
           │classify() │
           └──────────┘
    """
    print(graph)

print_dependency_graph()
print("Coding order: classify → compute → clean → format_report → run_pipeline → main")

**Figure 4.1** — A dependency graph showing coding order (bottom-up).

In [ ]:
# Real example — dependency tracking with a dict
dependencies = {
    "main": ["run_analysis"],
    "run_analysis": ["load_data", "process_data", "generate_report"],
    "process_data": ["clean_data", "compute_stats"],
    "generate_report": ["format_table", "format_summary"],
    "load_data": [],        # leaf
    "clean_data": [],       # leaf
    "compute_stats": [],    # leaf
    "format_table": [],     # leaf
    "format_summary": [],   # leaf
}

# Find leaf functions (implement these first)
leaves = [fn for fn, deps in dependencies.items() if not deps]
print("Leaf functions (code first):", leaves)

# Find root function (code last)
all_deps = set()
for deps in dependencies.values():
    all_deps.update(deps)
roots = [fn for fn in dependencies if fn not in all_deps]
print("Root function (code last) :", roots)

**Figure 4.2** — Using a dictionary to represent and analyze a dependency graph.

In [ ]:
# Identifying circular dependencies (a design mistake)
def check_circular(deps, func, visited=None):
    """Check if a function has circular dependencies."""
    if visited is None:
        visited = set()
    if func in visited:
        return True  # Circular!
    visited.add(func)
    for dep in deps.get(func, []):
        if check_circular(deps, dep, visited.copy()):
            return True
    return False

# Our clean dependencies have no cycles
for fn in dependencies:
    has_cycle = check_circular(dependencies, fn)
    status = "CYCLE!" if has_cycle else "OK"
    print(f"  {fn:20s} : {status}")

**Figure 4.3** — Checking for circular dependencies in the function graph.

---
## Part 5: Notebook Layout — CONFIG → Helpers → Pipeline → Output

A well-organized notebook follows a predictable structure:

| Section | Content | Example |
|---------|---------|---------|
| **CONFIG** | All constants and parameters | `THRESHOLD = 75` |
| **Imports** | Standard library imports | `import math` |
| **Helper functions** | Leaf-level utility functions | `def clean_data(...)` |
| **Pipeline functions** | Functions that compose helpers | `def run_analysis(...)` |
| **Main / Output** | Entry point and results | `main()` or pipeline calls |

This layout mirrors the dependency graph: leaves at the top, root at the bottom.

In [ ]:
# ═══════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════
BRIDGE_NAME = "Fatih Sultan Mehmet Köprüsü"
MAX_CAPACITY_TONS = 80
WARNING_PERCENT = 80
DATA_FILE = "traffic_data.csv"
INVALID_MARKERS = [None, -1, 0]
# ═══════════════════════════════════════════

print(f"Config loaded for: {BRIDGE_NAME}")

**Figure 5.1** — The CONFIG section at the top of the notebook.

In [ ]:
# ═══════════════════════════════════════════
# HELPER FUNCTIONS (leaf level)
# ═══════════════════════════════════════════

def is_valid_weight(w):
    """Check if a weight value is valid."""
    return w is not None and w not in INVALID_MARKERS and 0 < w < 100

def tons_to_kg(tons):
    """Convert tons to kilograms."""
    return tons * 1000

def kg_to_tons(kg):
    """Convert kilograms to tons."""
    return kg / 1000

def percentage(part, whole):
    """Calculate percentage safely."""
    if whole == 0:
        return 0.0
    return (part / whole) * 100

print("Helper functions loaded.")

**Figure 5.2** — Leaf-level helper functions (no dependencies on other custom functions).

In [ ]:
# ═══════════════════════════════════════════
# PIPELINE FUNCTIONS
# ═══════════════════════════════════════════

def prepare_data(raw_weights):
    """Filter and validate raw weight data."""
    return [w for w in raw_weights if is_valid_weight(w)]

def analyze_traffic(weights):
    """Compute traffic analysis from valid weights."""
    total = sum(weights)
    pct = percentage(total, MAX_CAPACITY_TONS)
    if pct > 100:
        status = "OVERLOADED"
    elif pct > WARNING_PERCENT:
        status = "WARNING"
    else:
        status = "SAFE"
    return {"total": total, "count": len(weights), "pct": pct, "status": status}

def format_traffic_report(analysis):
    """Format traffic analysis as a readable report."""
    lines = [
        f"Bridge: {BRIDGE_NAME}",
        f"Vehicles: {analysis['count']}",
        f"Total load: {analysis['total']:.1f} tons",
        f"Capacity usage: {analysis['pct']:.1f}%",
        f"Status: {analysis['status']}",
    ]
    return "\n".join(lines)

print("Pipeline functions loaded.")

**Figure 5.3** — Pipeline functions that compose helper functions.

In [ ]:
# ═══════════════════════════════════════════
# MAIN / OUTPUT
# ═══════════════════════════════════════════

# Simulated raw data (tons per vehicle)
raw_data = [3.5, None, 8.2, -1, 12.0, 15.0, 7.5, 4.2, 0, 22.5, 6.8]

valid_weights = prepare_data(raw_data)
analysis = analyze_traffic(valid_weights)
report = format_traffic_report(analysis)

print(report)

**Figure 5.4** — The output section: clean pipeline from data to result.

---
## Part 6: Eliminating Global Variables

**Global variables** are variables defined outside any function that are read (or worse, modified) inside functions.

### Why globals are bad:
- Functions become **unpredictable** (output depends on hidden state)
- Functions become **untestable** (can’t control inputs)
- Functions become **non-reusable** (tied to specific variable names)

### The fix:
- Pass values as **parameters**
- Return results instead of modifying globals
- Constants (ALL_CAPS) in a CONFIG cell are OK — they do not change

In [ ]:
# ❌ BAD: function depends on global variable
total = 0

def add_to_total(value):
    global total          # modifies global state!
    total += value

add_to_total(10)
add_to_total(20)
print(f"Total: {total}")  # 30 — but the function is not reusable or testable

**Figure 6.1** — A function that modifies global state — avoid this pattern.

In [ ]:
# ✅ GOOD: function takes input, returns output
def running_total(values):
    """Calculate the running total of a list."""
    total = 0
    result = []
    for v in values:
        total += v
        result.append(total)
    return result

values = [10, 20, 30, 15, 25]
totals = running_total(values)
print(f"Values : {values}")
print(f"Running: {totals}")

**Figure 6.2** — A pure function with no side effects: input in, output out.

In [ ]:
# ❌ BAD: multiple globals entangled
sensor_data = []
alert_count = 0
last_reading = None

def process_reading(value):
    global sensor_data, alert_count, last_reading
    sensor_data.append(value)
    last_reading = value
    if value > 50:
        alert_count += 1

process_reading(30)
process_reading(60)
process_reading(45)
print(f"Data: {sensor_data}, Alerts: {alert_count}, Last: {last_reading}")

**Figure 6.3** — Three global variables mutated inside a function — a maintenance nightmare.

In [ ]:
# ✅ GOOD: all state managed through parameters and return values
def process_all_readings(raw_readings, threshold=50):
    """Process readings and return results as a dict.

    Args:
        raw_readings: List of numeric readings.
        threshold: Alert threshold value.

    Returns:
        Dict with 'data', 'alert_count', 'last_reading' keys.
    """
    alerts = sum(1 for r in raw_readings if r > threshold)
    return {
        "data": list(raw_readings),
        "alert_count": alerts,
        "last_reading": raw_readings[-1] if raw_readings else None,
    }

result = process_all_readings([30, 60, 45])
print(f"Data: {result['data']}")
print(f"Alerts: {result['alert_count']}")
print(f"Last: {result['last_reading']}")

**Figure 6.4** — A clean function returning a results dictionary instead of mutating globals.

---
## Part 7: The main() Entry Point Pattern

In professional Python, the `main()` function is the single entry point that orchestrates everything:

```python
def main():
    data = load_data()
    processed = process(data)
    report = format_report(processed)
    print(report)

main()
```

### Benefits:
- All orchestration in **one place**
- Easy to see the **big picture**
- No global variables needed
- Easy to add parameters (e.g., `main(filename)` for different inputs)

In [ ]:
# The main() pattern for Mehmet's grade calculator

# CONFIG
GRADE_BOUNDARIES = [(90, "AA"), (85, "BA"), (80, "BB"),
                    (75, "CB"), (70, "CC"), (65, "DC"),
                    (60, "DD"), (0, "FF")]

def assign_letter_grade(score):
    """Convert numeric score to letter grade."""
    for boundary, grade in GRADE_BOUNDARIES:
        if score >= boundary:
            return grade
    return "FF"

def compute_class_stats(scores):
    """Compute statistics for a list of scores."""
    return {
        "count": len(scores),
        "mean": sum(scores) / len(scores),
        "min": min(scores),
        "max": max(scores),
        "pass_count": sum(1 for s in scores if s >= 60),
    }

def format_grade_report(students, stats):
    """Format the complete grade report."""
    lines = ["=" * 50, "  GRADE REPORT", "=" * 50]
    for name, score in students:
        grade = assign_letter_grade(score)
        lines.append(f"  {name:15s} : {score:3d} → {grade}")
    lines.append("-" * 50)
    lines.append(f"  Class mean    : {stats['mean']:.1f}")
    lines.append(f"  Pass rate     : {stats['pass_count']}/{stats['count']}")
    lines.append("=" * 50)
    return "\n".join(lines)

def main():
    """Entry point: orchestrate the grade report pipeline."""
    students = [
        ("Ahmet Yilmaz", 85),
        ("Zeynep Kaya", 72),
        ("Mehmet Demir", 91),
        ("Ayse Celik", 58),
        ("Ali Ozturk", 67),
        ("Fatma Sahin", 78),
    ]
    scores = [s for _, s in students]
    stats = compute_class_stats(scores)
    report = format_grade_report(students, stats)
    print(report)

main()

**Figure 7.1** — The `main()` pattern: one function orchestrates the entire pipeline.

In [ ]:
# main() with parameters — reusable for different inputs
def main_with_data(student_data):
    """Reusable entry point that accepts data as a parameter."""
    scores = [s for _, s in student_data]
    stats = compute_class_stats(scores)
    report = format_grade_report(student_data, stats)
    print(report)

# Different class data
section_b = [
    ("Burak Arslan", 95),
    ("Canan Korkmaz", 82),
    ("Deniz Acar", 70),
    ("Elif Yildiz", 63),
]

print("\n--- Section B ---\n")
main_with_data(section_b)

**Figure 7.2** — A parameterized `main()` that works with any input data.

---
## Part 8: Bottom-Up Implementation and Testing

**Bottom-up implementation** means:

1. Code and test **leaf functions** first (no dependencies)
2. Code and test **mid-level functions** that use leaves
3. Code the **pipeline/main** function last

At each level, **test immediately** with assertions before moving up.

### The workflow:
```
Code leaf → Test leaf → Code mid → Test mid → Code main → Integration test
```

This guarantees that when you wire things together, each piece already works.

In [ ]:
# Bottom-up: Level 1 — Leaf functions
def parse_sensor_value(raw_str):
    """Parse a sensor value string to float, return None if invalid."""
    try:
        val = float(raw_str)
        return val if -100 < val < 200 else None
    except (ValueError, TypeError):
        return None

def format_value(value, decimals=1, unit="°C"):
    """Format a numeric value with unit."""
    return f"{value:.{decimals}f}{unit}"

# TEST Level 1 immediately
assert parse_sensor_value("23.5") == 23.5
assert parse_sensor_value("abc") is None
assert parse_sensor_value(None) is None
assert parse_sensor_value("999") is None   # out of range

assert format_value(23.456) == "23.5°C"
assert format_value(1013, decimals=0, unit=" hPa") == "1013 hPa"

print("Level 1 tests passed ✓")

**Figure 8.1** — Coding and testing leaf functions before anything else.

In [ ]:
# Bottom-up: Level 2 — Mid-level functions using leaves
def process_raw_readings(raw_strings):
    """Parse and filter a list of raw sensor strings.

    Args:
        raw_strings: List of string values from sensor.

    Returns:
        List of valid float readings.
    """
    parsed = [parse_sensor_value(s) for s in raw_strings]
    return [v for v in parsed if v is not None]

def summarize_readings(readings):
    """Compute summary statistics for valid readings.

    Args:
        readings: Non-empty list of floats.

    Returns:
        Dict with mean, min, max, count.
    """
    return {
        "mean": sum(readings) / len(readings),
        "min": min(readings),
        "max": max(readings),
        "count": len(readings),
    }

# TEST Level 2
test_raw = ["23.5", "bad", "25.0", None, "999", "22.1"]
result = process_raw_readings(test_raw)
assert result == [23.5, 25.0, 22.1], f"Got {result}"

stats = summarize_readings([10, 20, 30])
assert stats["mean"] == 20.0
assert stats["min"] == 10
assert stats["max"] == 30
assert stats["count"] == 3

print("Level 2 tests passed ✓")

**Figure 8.2** — Mid-level functions tested after leaf functions are confirmed working.

In [ ]:
# Bottom-up: Level 3 — Pipeline / main function
def sensor_report(raw_data, station_name="Unnamed"):
    """Generate a complete sensor report from raw data.

    Args:
        raw_data: List of raw string readings.
        station_name: Name of the weather station.

    Returns:
        Formatted report string.
    """
    readings = process_raw_readings(raw_data)
    if not readings:
        return f"Station {station_name}: No valid readings!"

    stats = summarize_readings(readings)
    lines = [
        f"Station: {station_name}",
        f"  Valid readings: {stats['count']} / {len(raw_data)}",
        f"  Mean: {format_value(stats['mean'])}",
        f"  Range: {format_value(stats['min'])} – {format_value(stats['max'])}",
    ]
    return "\n".join(lines)

# Integration test
raw = ["22.1", "bad", "25.3", "999", None, "23.7", "21.9"]
report = sensor_report(raw, station_name="Istanbul-Kadikoy")
print(report)
print()

# Edge case: no valid data
empty_report = sensor_report(["bad", None, "9999"])
print(empty_report)

**Figure 8.3** — The pipeline function composes tested building blocks.

In [ ]:
# Final summary: the complete bottom-up workflow
print("""
Bottom-Up Implementation Workflow:
══════════════════════════════════
1. Write leaf functions    → parse_sensor_value, format_value
   Test immediately        → 6 assertions ✓

2. Write mid-level funcs   → process_raw_readings, summarize_readings
   Test immediately        → 5 assertions ✓

3. Write pipeline/main     → sensor_report
   Integration test        → 2 test cases ✓

Total: 3 levels, 11+ tests, zero globals, every function < 10 lines.
""")

**Figure 8.4** — Summary of the bottom-up workflow.

---
## Exercises

Complete each exercise in the code cell below it. Each cell is marked with `# ✏️ [EXn]` so the submission system can find your answers.

**Exercise 1.** Write a function `is_valid_temperature(t)` that returns `True` if `t` is a number between -50 and 60. Test it with 5 assertions including edge cases and invalid types.

In [ ]:
# ✏️ [EX1]


**Exercise 2.** Write a function `filter_valid(data, validator)` that takes a list and a validator function, and returns only items where the validator returns `True`. Test with `is_valid_temperature`.

In [ ]:
# ✏️ [EX2]


**Exercise 3.** Decompose this problem into 3-4 functions: "Given a list of exam scores, compute the average, find the highest and lowest, and print a formatted report." Write the function stubs (name + docstring + `pass`) first.

In [ ]:
# ✏️ [EX3]


**Exercise 4.** Draw (as text) the dependency graph for your Exercise 3 functions. Print it using a multi-line string.

In [ ]:
# ✏️ [EX4]


**Exercise 5.** Implement the functions from Exercise 3, following bottom-up order. Test each function with at least 2 assertions before moving to the next level.

In [ ]:
# ✏️ [EX5]


**Exercise 6.** Refactor this code to eliminate global variables. All values should be passed as parameters:
```python
data = [15, 22, 18, 30, 25]
threshold = 20
results = []
for d in data:
    if d > threshold:
        results.append(d)
print(len(results), sum(results))
```

In [ ]:
# ✏️ [EX6]


**Exercise 7.** Write a function `create_student_report(name, scores)` that returns a formatted string with the student’s name, scores, average, and letter grade. Use at least 2 helper functions. Test with Ayse’s scores: `[78, 85, 92, 70, 88]`.

In [ ]:
# ✏️ [EX7]


**Exercise 8.** Organize a complete notebook-style solution with CONFIG → helpers → pipeline → output for a **resistance calculator**: given a list of resistor values and a configuration (series/parallel), compute total resistance. Use at least 3 functions.

In [ ]:
# ✏️ [EX8]


**Exercise 9.** Write a `main()` function that orchestrates a **unit converter**: it takes a list of `(value, from_unit, to_unit)` tuples and prints a conversion table. Support at least 3 unit pairs (e.g., km↔mi, kg↔lb, C↔F).

In [ ]:
# ✏️ [EX9]


**Exercise 10.** Given this 30-line monolithic code, decompose it into at least 4 functions. Rewrite it using the main() pattern.

```python
nums = [12, 45, 7, 23, 89, 34, 56, 11, 78, 43]
even = []
odd = []
for n in nums:
    if n % 2 == 0:
        even.append(n)
    else:
        odd.append(n)
print("Even:", even)
print("Odd:", odd)
print("Even sum:", sum(even))
print("Odd sum:", sum(odd))
print("Even avg:", sum(even)/len(even))
print("Odd avg:", sum(odd)/len(odd))
if sum(even) > sum(odd):
    print("Even numbers have higher total")
else:
    print("Odd numbers have higher total")
```

In [ ]:
# ✏️ [EX10]


**Exercise 11.** Write a bottom-up implementation of a **simple calculator** pipeline: `parse_expression(expr_str)` → `evaluate(parsed)` → `format_result(value)`. Support +, -, *, / with two operands (e.g., `"3 + 5"`). Test each level.

In [ ]:
# ✏️ [EX11]


**Exercise 12.** Define I/O contracts (as docstrings with Args/Returns/Raises) for these function signatures, then implement and test each:
- `normalize(values)` — scale values to 0-1 range
- `z_score(value, mean, std)` — compute z-score
- `classify_z(z)` — classify as "low"/"normal"/"high"

In [ ]:
# ✏️ [EX12]


**Exercise 13.** Create a complete **sensor data pipeline** with CONFIG cell, 4+ helper functions, a `main()` function, and at least 8 assertions. The pipeline should: read raw data (simulated), validate, compute statistics, classify status, and format a report. Use Turkish city names.

In [ ]:
# ✏️ [EX13]


**Exercise 14.** Identify what is wrong with this code and refactor it. List each problem you find as a comment.

```python
x = []
y = 0
def f(v):
    global x, y
    x.append(v)
    y = y + v
    if y / len(x) > 50:
        print("high")
    return y / len(x)
f(40)
f(60)
f(55)
print(x, y)
```

In [ ]:
# ✏️ [EX14]


**Exercise 15 — Capstone.** Decompose and implement a **mini grade management system**. Given a list of students (name, midterm, final, homework scores), the system should:
1. Validate all scores (0-100)
2. Compute weighted averages (midterm 30%, final 40%, homework 30%)
3. Assign letter grades
4. Compute class statistics
5. Generate a formatted report

Use the full Decomposition Recipe. Implement bottom-up with tests at each level. Use a CONFIG cell, at least 5 functions, and a `main()` entry point.

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

This week you learned the **Decomposition Recipe** — a systematic approach to breaking complex problems into small, testable functions. You practiced eliminating globals, drawing dependency graphs, and building code bottom-up.

Next week we will explore **Strings, Formatting, and Text Processing** — essential tools for cleaning data, generating reports, and handling real-world text. You will see how all the modular design skills from this week apply to text-processing pipelines.

---
## 📮 Submission

**STEP 1 —** Fill in your information below and run the cell to verify.

In [ ]:
STUDENT_ID = ""
STUDENT_NAME = ""
STUDENT_EMAIL = ""
CLASS_CODE = ""

import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID): _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2: _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16: _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4: _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors: print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\U0001f449 Now run the NEXT cell to submit.")

**STEP 2 —** Run the cell below to submit your work. Make sure you have executed all exercise cells first.

In [ ]:
# STEP 2 — Submit your work
import urllib.request, json as _json, re as _re2

_WEEK = "Week_03"
_URL = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
_SOURCE = "cp2-notebook"

# --- collect answers from executed cells ---
_answers = {}
try:
    _hist = list(In)
except NameError:
    _hist = []

for _i, _c in enumerate(_hist):
    _m = _re2.search(r"#\s*\u270f\ufe0f\s*\[EX(\d+)\]", str(_c))
    if _m:
        _answers[f"ex{_m.group(1)}"] = str(_c)

_payload = _json.dumps({
    "week": _WEEK,
    "source": _SOURCE,
    "studentId": STUDENT_ID,
    "studentName": STUDENT_NAME,
    "studentEmail": STUDENT_EMAIL,
    "classCode": CLASS_CODE,
    "answers": _answers
}).encode()

_req = urllib.request.Request(_URL, data=_payload,
                              headers={"Content-Type": "application/json"})
try:
    _resp = urllib.request.urlopen(_req, timeout=15)
    _body = _json.loads(_resp.read().decode())
    if _body.get("status") == "ok":
        print(f"\u2705 Submitted {len(_answers)} answer(s) for {_WEEK}.")
        print(f"   Timestamp: {_body.get('timestamp', 'n/a')}")
    else:
        print("\u26a0\ufe0f  Server response:", _body)
except Exception as _ex:
    print(f"\u274c Submission failed: {_ex}")
    print("Try again or contact your instructor.")